# ML-02 — Research Question and Provisional Lane

This notebook records my provisional framing and reproducible aggregate evidence from the starter dataset. My lane can change until the end of Week 4. Run all cells from a checkout of this repository, either at its root or in `work/notebooks/`.

## 1. My lane (or freestyle) and why

**Provisional lane: Lane 2 — Refresh / Content Opportunity Scoring.** I want to help a content editor choose which pages deserve review when time is limited. The starter data offers page-level search demand, recent impression changes, and update age. Section 3 shows a substantial pool of declining pages with demand, but very few pages meet the starter's stale-visible rule. This makes prioritizing review worth investigating; it does not establish that a rewrite would help. Over the next seven weeks I will check data quality, audit signals, build an understandable baseline, compare a more complex method only if warranted, validate it, and produce a review queue. If suitable future-window data is unavailable, the result will remain descriptive decision support.

Reference: `docs/ml-intern-dataset-and-lane-guide.md`, Lane 2.

In [1]:
LANE = "Refresh / Content Opportunity Scoring"
REVIEW_BUDGET = 20  # Provisional pages reviewed per client per cycle.
MIN_DEMAND_IMPRESSIONS = 100
STALE_DAYS = 180
STALE_VISIBLE_IMPRESSIONS = 500
print(f"Provisional lane: {LANE}")
print(f"Provisional review budget: {REVIEW_BUDGET} pages per client per cycle")

Provisional lane: Refresh / Content Opportunity Scoring
Provisional review budget: 20 pages per client per cycle


## 2. The question: decision, action, cost of a wrong call

**Search question:** Among pages with measurable search demand, which should a content editor review first for possible refresh or monitoring, and can an explainable score prioritize them better than simply reviewing the highest-impression pages?

**Unit of analysis:** One pseudonymized content item (page) within one client at the starter export snapshot. Later warehouse analysis would use one page at a decision date, with features measured before that date. Rank within each client so a large client does not consume everyone's review budget.

**Decision and output:** Choose the next 20 pages per client to inspect per review cycle (a provisional capacity assumption). Produce a ranked queue containing pseudonymized page IDs, priority scores, measurement coverage, reason codes, and suggested review actions. A priority score is not a probability of refresh success.

**Who acts and how:** A content editor checks the actual page and search context, then decides whether to revise outdated information, investigate a search-snippet issue, or monitor the page. Metrics alone do not justify automatic rewriting, deletion, or pruning; this dataset does not contain page text or actual queries.

**Cost of a wrong call:** A false positive wastes review time and may encourage unnecessary edits to a useful page. A false negative leaves a potentially important decline unreviewed, with possible missed traffic and delayed diagnosis. Editor hours and revenue loss are not measured here, so I will not invent monetary costs. Minimum demand, a fixed review budget, and visible reasons help limit these risks.

**Why data or ML helps:** Data can reduce manual sorting across thousands of pages by combining demand, trend, freshness, and measurement reliability. Start with a transparent rule and a highest-impressions baseline. ML earns a place only if pre-decision signals improve held-out prioritization enough to justify the added complexity. A useful rule or an inconclusive result is acceptable: the goal is a better review decision, not a trained model.

**Task and metric:** Ranking/scoring. Today I can measure a baseline's share of pages with the observed `trend_direction == 'down'` flag, but this is a current-window proxy, not verified editorial need. Section 4 computes it. For a later predictive comparison, my provisional target is a greater-than-20% fall in next-30-day impressions versus preceding-30-day impressions, among pages with at least 100 preceding-30-day impressions. Use prior-90-day features ending at the decision date. Evaluate precision@20 per client on later periods and held-out clients against the highest-demand baseline; report coverage, base rate, and human review of reasons. Clients with fewer than 20 eligible pages use their actual queue length and are reported separately. Check history coverage and threshold sensitivity before fixing this target. Even future decline measures review risk, not the benefit of editing.

In [2]:
FRAME = {
    "unit": "page within client at a decision date",
    "actor": "content editor",
    "decision": "which pages to review first",
    "action": "inspect, then decide whether to refresh or monitor",
    "metric": "precision@20 against an explicitly named decline proxy",
}
assert REVIEW_BUDGET > 0 and MIN_DEMAND_IMPRESSIONS > 0
assert all(FRAME.values())
print("Decision, actor, action, grain, and evaluation metric are recorded.")

Decision, actor, action, grain, and evaluation metric are recorded.


## 3. Quick look at the data (2-3 real numbers)

The cell below directly loads `data/raw/content_refresh_anonymized.csv` and shows three measurements:

1. Inventory size and client coverage establish the scale of review.
2. Pages with `trend_direction == 'down'` and at least 100 impressions in 90 days measure the pool with recent decline and some observed demand. The supplied flag compares the last 30 days with the preceding 30 days; it is not a future outcome.
3. Pages with at least 180 days since their last update and at least 500 impressions in 90 days measure the scope of the starter's stale-visible rule.

These groups can overlap and should not be added as if disjoint. The 100-impression floor screens out small 90-day totals, but does not guarantee reliable 30-day trends. Age and decline alone do not show that editing will help.

In [3]:
from pathlib import Path
import pandas as pd

# Locate the CSV from the repository root or a nested notebook directory.
repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "data/raw/content_refresh_anonymized.csv").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Open this notebook inside a checkout containing the starter CSV.")
data = pd.read_csv(repo_root / "data/raw/content_refresh_anonymized.csv")
required = {"content_id", "client_id", "impressions_90d", "trend_direction",
            "days_since_last_update", "content_age_days", "avg_position", "ctr"}
assert required.issubset(data.columns), "Required starter columns are missing."
assert not data[["client_id", "content_id"]].isna().any().any()
assert data["content_id"].is_unique, "Check page grain before counting."
assert len(data) > 0

has_demand = data["impressions_90d"].ge(MIN_DEMAND_IMPRESSIONS)
declining_with_demand = has_demand & data["trend_direction"].eq("down")
stale_visible = (
    data["days_since_last_update"].ge(STALE_DAYS)
    & data["impressions_90d"].ge(STALE_VISIBLE_IMPRESSIONS)
)
print(f"1. Inventory: {len(data):,} unique pages, {data.client_id.nunique():,} clients, "
      f"{len(data.columns)} columns; duplicate page IDs: 0.")
print(f"2. Declining with demand: {declining_with_demand.sum():,} / {len(data):,} pages "
      f"({100 * declining_with_demand.mean():.2f}%).")
print(f"3. Stale and visible: {stale_visible.sum():,} / {len(data):,} pages "
      f"({100 * stale_visible.mean():.2f}%).")

1. Inventory: 30,000 unique pages, 32 clients, 44 columns; duplicate page IDs: 0.
2. Declining with demand: 13,152 / 30,000 pages (43.84%).
3. Stale and visible: 17 / 30,000 pages (0.06%).


**What the counts suggest:** The snapshot contains **30,000 unique pages across 32 clients**. **13,152 pages (43.84%)** have the supplied decline flag and at least 100 impressions over 90 days. Only **17 pages (0.06%)** meet the stale-visible thresholds. This supports investigating review priority within a large decline-with-demand pool instead of relying on staleness alone. It does not show that all 13,152 pages require refresh, or that this slice represents every client's full inventory.

## 4. Careful words: what I can and can't claim

I can say **we observed** these aggregate counts in the starter slice and they suggest a useful **decision-support** question. I can compare transparent rules against a clearly named proxy with coverage reported. I cannot yet claim a useful predictor, better editorial decisions, future recovery, revenue gain, a causal refresh effect, or knowledge of Google's algorithm. There are no editor judgments, interventions, or future outcomes here.

The supplied `trend_direction` is derived from `trend_pct` and the 30-day impression comparison. Reproducing this flag learns a rule, not future decline or refresh value. For any supervised starter-proxy experiment, exclude `trend_direction`, `trend_pct`, and their constituent 30-day impression fields from features. Even trailing-90-day totals overlap the proxy period, so client holdout alone would not make that experiment a forecasting test. Future-window claims require non-overlapping warehouse feature/outcome windows, per-client history checks, and a sealed final month during development.

IDs are for grouping, joining, splitting, and locating recommendations, never predictive features. Rates already represent percentages (`ctr = 0.76` means 0.76%). Position zero means no measurement. Keep missing keyword/word-count data distinguishable from measured zeros. Scroll and AI traffic rates can exceed 100. For warehouse work, filter availability flags with `IS TRUE`; missing tracking is not zero traffic.

**Measurable baseline today:** Rank eligible pages within each client by 90-day impressions, then compute the selected pages' current decline-flag share. The flag is used only for diagnosis, not to choose the baseline queue. This is a same-snapshot diagnostic, not out-of-sample validation. An eligible-pool base rate provides context; neither result establishes refresh benefit.

In [4]:
eligible = data.loc[has_demand].copy()
# ID is only a deterministic tie break, not a predictive feature.
queue = (
    eligible.sort_values(["client_id", "impressions_90d", "content_id"],
                         ascending=[True, False, True])
    .groupby("client_id", sort=False).head(REVIEW_BUDGET)
)
queue = queue.assign(observed_down=queue["trend_direction"].eq("down"))
per_client = queue.groupby("client_id")["observed_down"].agg(["mean", "size"])
full_queues = per_client.loc[per_client["size"].eq(REVIEW_BUDGET)]
short_queues = per_client.loc[per_client["size"].lt(REVIEW_BUDGET)]
assert not full_queues.empty, "Not enough eligible pages for precision@20."
print(f"Demand-only baseline: {len(queue):,} selected pages across {len(per_client)} clients.")
print(f"Current-proxy macro precision@{REVIEW_BUDGET}: "
      f"{100 * full_queues['mean'].mean():.2f}% "
      f"({len(full_queues)} clients with full queues).")
print(f"Short queues reported separately: {len(short_queues)} clients.")
if not short_queues.empty:
    print(f"Short-queue mean current decline share: {100 * short_queues['mean'].mean():.2f}%.")
base_rates = (
    eligible.assign(observed_down=eligible["trend_direction"].eq("down"))
    .groupby("client_id")["observed_down"].mean()
)
print(f"Eligible-pool macro current decline share (same full-queue clients): "
      f"{100 * base_rates.loc[full_queues.index].mean():.2f}%.")
print("Diagnostic only: a current trend flag, not future decline or refresh payoff.")

Demand-only baseline: 528 selected pages across 30 clients.
Current-proxy macro precision@20: 47.29% (24 clients with full queues).
Short queues reported separately: 6 clients.
Short-queue mean current decline share: 41.94%.
Eligible-pool macro current decline share (same full-queue clients): 58.66%.
Diagnostic only: a current trend flag, not future decline or refresh payoff.


## 5. Self-check

- [x] I chose a predefined lane and explained why it is provisional.
- [x] I named the question, page/client grain, decision, output, editor action, and costs of errors.
- [x] Code loads the starter CSV and reports three aggregate measurements with explicit thresholds.
- [x] I explained why a transparent rule comes first and when ML would earn its place.
- [x] A measurable baseline diagnostic is distinguished from future validation and editorial usefulness.
- [x] The notebook runs top to bottom without errors and its cell outputs are saved.
- [x] Outputs contain aggregate metrics only: no client names, page URLs, private queries, page text, or credentials.
- [x] Claims use observed, measured, directional, and decision-support language; refresh benefit is unproven.
- [x] The notebook lives at `work/notebooks/w01_research_question.ipynb` and is included in the task commit.

**Submission repository:** [FlyrankAI-Internship](https://github.com/vibhanshu-s/FlyrankAI-Internship). Submit the repository URL on the card after the executed notebook commit is published.